In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from src.fluid      import Fluid
from src.interpolator import LinearInterpolator
from src.reservoir  import ResProps, Reservoir
from src.pipe       import Pipe
from src.well       import Well
from src.compressor import DCS
from src.simulator  import FieldSimulator

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True

ModuleNotFoundError: No module named 'scipy'

In [ ]:
# ← Sustituir con los valores individuales de hw2_data.csv
M     = 0.01604   # kg/mol  (ejemplo: metano puro)
rho_c = 0.6790    # kg/m³ a condiciones estándar
xa    = 0.01      # fracción molar N₂
xy    = 0.01      # fracción molar CO₂
T_res = 310.0     # K — temperatura del yacimiento

fluid = Fluid(M=M, rho_c=rho_c, xa=xa, xy=xy, T=T_res)

In [ ]:
import math

# Yacimiento
P0    = 100.0                          # atm — presión inicial
V_res = math.pi * 500**2 * 25         # m³

# Pozos: [k, h, re, rw, L, H, D, roughness]
well_params = [
    dict(k=50, h=25, re=500, rw=0.1, L=2000, H=1800, D=0.062, roughness=0.000046),
    dict(k=50, h=25, re=500, rw=0.1, L=2500, H=1900, D=0.062, roughness=0.000046),
    dict(k=50, h=25, re=500, rw=0.1, L=1800, H=1600, D=0.073, roughness=0.000046),
]

# Шлейф
shlyf_params = dict(L=5000, D=0.200, roughness=0.000046)

# ДКС
CR     = 1.5
P_line = 5.0    # atm
q_ext  = 500.0  # ст.м³/сут

In [ ]:
resprops = ResProps(P=P0, V=V_res, T=T_res)
reservoir = Reservoir(resprops=resprops, fluid=fluid)

pipes = [Pipe(L=wp['L'], D=wp['D'], roughness=wp['roughness'],
              fluid=fluid, vertical_depth=wp['H'])
         for wp in well_params]

wells = [Well(fluid=fluid, k=wp['k'], h=wp['h'],
              re=wp['re'], rw=wp['rw'], pipe=pipe)
         for wp, pipe in zip(well_params, pipes)]

shlyf = Pipe(**shlyf_params, fluid=fluid, vertical_depth=0.0)
dcs   = DCS(CR=CR, P_line=P_line, q_ext=q_ext)

sim = FieldSimulator(reservoir=reservoir, wells=wells, shlyf=shlyf, dcs=dcs)

Sección 1 — Modelo PVT
Objetivo
Graficar Z(P), Bg(P), μ(P), ρ(P) en rango 1–200 atm. Comparar ρ_ideal vs ρ_real.

In [ ]:
P_range = np.linspace(1, 200, 200)

Z_vals   = [fluid.z(P)   for P in P_range]
Bg_vals  = [fluid.bg(P)  for P in P_range]
mu_vals  = [fluid.mu(P)  for P in P_range]
rho_vals = [fluid.ro(P)  for P in P_range]

# ρ ideal: Z = 1 siempre
R = 8.314
rho_ideal = [P * 101325 * M / (1.0 * R * T_res) for P in P_range]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].plot(P_range, Z_vals);   axes[0,0].set(title='Z(P)', xlabel='P [atm]', ylabel='Z')
axes[0,1].plot(P_range, Bg_vals);  axes[0,1].set(title='Bg(P)', xlabel='P [atm]', ylabel='Bg [м³/м³]')
axes[1,0].plot(P_range, mu_vals);  axes[1,0].set(title='μ(P)', xlabel='P [atm]', ylabel='μ [cP]')
axes[1,1].plot(P_range, rho_vals, label='Real')
axes[1,1].plot(P_range, rho_ideal, '--', label='Ideal')
axes[1,1].set(title='ρ(P)', xlabel='P [atm]', ylabel='ρ [кг/м³]')
axes[1,1].legend()

plt.tight_layout()
plt.show()

Sección 2 — Modelo de yacimiento (IPR)
Objetivo
Curva IPR para cada pozo a P_res = 100 atm.

In [ ]:
P_res_test = 100.0
P_bhp_range = np.linspace(0, P_res_test, 100)

fig, ax = plt.subplots()
for i, well in enumerate(wells):
    q_ipr = [well.q(P_res_test, Pb) for Pb in P_bhp_range]
    ax.plot(q_ipr, P_bhp_range, label=f'Скважина {i+1}')

ax.set(xlabel='q [ст.м³/сут]', ylabel='P_bhp [atm]',
       title=f'IPR при P_res = {P_res_test} atm')
ax.legend()
plt.show()

Sección 3 — Hidráulica (VLP y λ(Re))
Objetivo
Curvas VLP de las 3 скважины y el шлейф. Dependencia λ(Re) por Colebrook-White.

In [ ]:
P_man_test = 10.0
q_range = np.linspace(10, 2000, 100)

# VLP скважин
fig, ax = plt.subplots()
for i, (well, pipe) in enumerate(zip(wells, pipes)):
    P_bhp_vlp = [P_man_test + pipe.dp(P_man_test, q).dP for q in q_range]
    ax.plot(q_range, P_bhp_vlp, label=f'VLP Скважина {i+1}')
ax.set(xlabel='q [ст.м³/сут]', ylabel='P_bhp [atm]', title='Кривые VLP')
ax.legend(); plt.show()

# VLP шлейфа
fig, ax = plt.subplots()
q_shlyf = np.linspace(100, 6000, 100)
dP_shlyf = [shlyf.dp(dcs.P_in(), q).dP for q in q_shlyf]
ax.plot(q_shlyf, dP_shlyf)
ax.set(xlabel='q_total [ст.м³/сут]', ylabel='ΔP [atm]', title='Гидравлика шлейфа')
plt.show()

# λ(Re)
Re_range = np.logspace(2, 7, 200)
# Usar pipe del pozo 1 como referencia
lambda_vals = [pipes[0].f_lambda(Re) for Re in Re_range]
fig, ax = plt.subplots()
ax.loglog(Re_range, lambda_vals)
ax.axvline(2300, color='r', linestyle='--', label='Re = 2300')
ax.set(xlabel='Re', ylabel='λ', title='Коэффициент трения по Колбруку–Уайту')
ax.legend(); plt.show()

Sección 4 — Punto de operación (IPR + VLP)
Objetivo
Para cada поzo: graficar IPR y VLP en el mismo eje, marcar punto de operación.

In [ ]:
states_0 = sim.solve(P_res=P0)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (well, pipe, ax) in enumerate(zip(wells, pipes, axes)):
    P_man_sol = states_0[f'well_{i+1}'].P_in
    q_sol     = states_0[f'well_{i+1}'].q_std
    P_bhp_sol = states_0[f'well_{i+1}'].P_out

    q_plot    = np.linspace(10, 1500, 200)
    q_ipr     = [well.q(P0, Pb) for Pb in np.linspace(0, P0, 200)]
    P_bhp_ipr = np.linspace(0, P0, 200)
    P_bhp_vlp = [P_man_sol + pipe.dp(P_man_sol, q).dP for q in q_plot]

    ax.plot(q_ipr, P_bhp_ipr, label='IPR')
    ax.plot(q_plot, P_bhp_vlp, label='VLP')
    ax.scatter([q_sol], [P_bhp_sol], color='red', zorder=5, label=f'Рабочая точка\nq={q_sol:.0f} м³/сут')
    ax.set(title=f'Скважина {i+1}', xlabel='q [ст.м³/сут]', ylabel='P_bhp [atm]')
    ax.legend()

plt.tight_layout(); plt.show()

Sección 5 — Dinámica (180 días)
Objetivo
Graficar P_res(t), P_man(t), q1(t), q2(t), q3(t), Gp(t).

In [ ]:
# Reiniciar yacimiento al estado inicial
reservoir.resprops.P = P0

df = sim.run(N_days=180, dt=1.0)

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

axes[0,0].plot(df['t'], df['P_res']); axes[0,0].set(ylabel='P_res [atm]', title='Пластовое давление')
axes[0,1].plot(df['t'], df['P_man']); axes[0,1].set(ylabel='P_man [atm]', title='Давление на манифолде')
axes[1,0].plot(df['t'], df['q1'], label='q1')
axes[1,0].plot(df['t'], df['q2'], label='q2')
axes[1,0].plot(df['t'], df['q3'], label='q3')
axes[1,0].set(ylabel='q [ст.м³/сут]', title='Дебиты скважин'); axes[1,0].legend()
axes[1,1].plot(df['t'], df['q_total']); axes[1,1].set(ylabel='q_total [ст.м³/сут]', title='Суммарный дебит')
axes[2,0].plot(df['t'], df['Gp']); axes[2,0].set(ylabel='Gp [тыс.ст.м³]', title='Накопленная добыча')
axes[2,1].axis('off')  # reservado

for ax in axes.flat:
    ax.set_xlabel('t [сут]')
plt.tight_layout(); plt.show()

Sección 6 — Influencia del ДКС
Objetivo
Comparar dinámicas para CR ∈ {1.0, 1.5, 2.0, 3.0}.

In [ ]:
CR_values = [1.0, 1.5, 2.0, 3.0]
results_cr = {}

for cr in CR_values:
    reservoir.resprops.P = P0     # reiniciar yacimiento
    dcs.CR = cr
    df_cr = sim.run(N_days=180, dt=1.0)
    results_cr[cr] = df_cr

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for cr, df_cr in results_cr.items():
    axes[0].plot(df_cr['t'], df_cr['P_res'],   label=f'CR={cr}')
    axes[1].plot(df_cr['t'], df_cr['q_total'], label=f'CR={cr}')
    axes[2].plot(df_cr['t'], df_cr['Gp'],      label=f'CR={cr}')

axes[0].set(title='P_res(t)', xlabel='t [сут]', ylabel='P_res [atm]')
axes[1].set(title='q_total(t)', xlabel='t [сут]', ylabel='q [ст.м³/сут]')
axes[2].set(title='Gp(t)', xlabel='t [сут]', ylabel='Gp [тыс.ст.м³]')
for ax in axes: ax.legend()
plt.tight_layout(); plt.show()

# Restaurar CR original
dcs.CR = CR

Sección 7 — Calibración
Objetivo
Ajustar coeficiente de productividad C minimizando RMSE vs datos reales.
Calcular R² y RMSE tras la calibración.

In [ ]:
from scipy.optimize import minimize

# Cargar datos de campo
df_fact = pd.read_csv('adapt_gdi_11-2025.csv')
# Ajustar nombres de columnas según el archivo real
q_fact   = df_fact['q'].values
P_res_fact = df_fact['P_res'].values
P_bhp_fact = df_fact['P_bhp'].values

def model_q(C_scale):
    """
    Simula débito con coeficiente de productividad escalado.
    C_scale multiplica k de todas las скважины.
    """
    qs = []
    for P_r, P_b in zip(P_res_fact, P_bhp_fact):
        q_sim = 0
        for well in wells:
            # Escalar k temporalmente
            k_orig = well.k
            well.k = well.k * C_scale[0]
            q_sim += well.q(P_r, P_b)
            well.k = k_orig
        qs.append(q_sim)
    return np.array(qs)

def rmse(C_scale):
    q_sim = model_q(C_scale)
    return np.sqrt(np.mean((q_sim - q_fact)**2))

result = minimize(rmse, x0=[1.0], method='Nelder-Mead')
C_opt = result.x[0]
print(f"C_scale óptimo: {C_opt:.4f}")

q_sim_opt = model_q([C_opt])

# Métricas
ss_res = np.sum((q_fact - q_sim_opt)**2)
ss_tot = np.sum((q_fact - np.mean(q_fact))**2)
R2   = 1 - ss_res / ss_tot
RMSE = np.sqrt(np.mean((q_fact - q_sim_opt)**2))
print(f"R² = {R2:.4f},  RMSE = {RMSE:.2f} ст.м³/сут")

# Gráfico факт vs модель
fig, ax = plt.subplots()
ax.scatter(range(len(q_fact)), q_fact,    label='Факт', alpha=0.7)
ax.plot(range(len(q_sim_opt)), q_sim_opt, label='Модель', color='red')
ax.set(xlabel='Точка', ylabel='q [ст.м³/сут]',
       title=f'Калибровка: R²={R2:.3f}, RMSE={RMSE:.1f}')
ax.legend(); plt.show()